<a href="https://colab.research.google.com/github/tmacpherson6/investment_advisor_dashboard_capstone/blob/master/Kristine_SCF_Feature_Selection_R.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Survey of Consumer Finances (SCF) Feature Selection**

This notebook will utilize the **glmnet** package in R to run LASSO regression on the SCF data for feature selection.

The SCF addresses item nonresponse through **multiple imputation**. Each release includes five implicates — plausible, complete versions of the dataset with different imputed values for missing items. We will run LASSO on each of the 5 imputations utilizing the survey weights, and use variables present in at least 4 of the imputations to create the final regression model.

### **Packages**

In [ ]:
install.packages("glmnet")
install.packages("survey")
install.packages("readxl")
install.packages("tidyverse")

library(glmnet)
library(survey)
library(readxl)
library(tidyverse)

### **Filter out variables**

Download variable list from the Survey of Consumer Finances website:

In [2]:
temp_var_df <- tempfile(fileext = ".xlsx")

download.file("https://www.federalreserve.gov/econres/files/2022map.xlsx", destfile = temp_var_df, mode = "wb"
)

var_tbl <- read_xlsx(temp_var_df, col_types = "text")
var_df <- as.data.frame(var_tbl)

file.remove(temp_var_df)
names(var_df) <- tolower(names(var_df))
var_df[[1]] <- tolower(as.character(var_df[[1]]))

New names:
• `` -> `...2`
• `6` -> `6...4`
• `6` -> `6...5`


[1] TRUE

We want to remove questions that are not broadly applicable or difficult to answer. For example, we will filter out questions that were asked to the respondents' spouse or partner, and instead focus on questions only asked to the respondent. Many questions include followup questions (denoted with an underscore followed by a number) that we will filter out as well.

In [3]:
original_vars = var_df[[1]]

removalwords <- c("covid", "recoded", "s/p", "person_", "inst_", "_vac", "zz", "spouse", "person4", "person5")

var_df <- var_df %>%
  filter(!startsWith(as.character(.[[1]]), "j")) %>%
  filter(!str_detect(str_to_lower(coalesce(as.character(.[[2]]), "")),
                     str_c(removalwords, collapse = "|"))) %>%
  filter(!str_detect(coalesce(as.character(.[[2]]), ""),"_[2-9]\\d*"))

In [4]:
var_df[sample(nrow(var_df), 10), ]

,j14,...2,6.1,6...4,6...5
,<chr>,<chr>,<chr>,<chr>,<chr>
433,x3451,X3451_Q545A3: BUS_LLC: HAVE LLCs,14.2,14,27268
1375,x11011,X11011_Q1063A1B1M3: R_P_JOB_1: BEN TYPE AVAIL 3,14.2,14,50508
1142,x7491,X7491_Q1700A: EXPENSES HIGH/LOW/NORMAL,14.2,14,43340
360,x3125,X3125_Q512A1: ACT_MAN_BUS_1: R OWE BUS MONEY?,14.2,14,25504
308,x2722,X2722_Q724A1: OTH_LN_1: YR EXPECT REPAY,14.2,14,23852
941,x6830,X6830_Q908A2M3: OTHER,14.2,14,38006
650,x5320,X5320_Q1313A1: P_REC_1: INCREASE W/COST OF LIVE?,14.2,14,32238
543,x4018,X4018_Q954: AMT OWED TO R,14.2,14,29466
894,x6715,X6715_Q1042A2M3: PROFIT SHARING,14.2,14,37250


### **Prepare the data for regression**

In [5]:
scf_list <- read.csv('/content/scf_list.csv')

Remove questions related to financial risk except for **x7557**. This is the variable we are interested in predicting, and represents a respondents' willingness to take financial risk on a scale of 0 - 10.

In [6]:
scf_list <- scf_list %>% select(-any_of(c("x3014", "yesfinrisk", "nofinrisk")))

In [8]:
original_vars <- tolower(as.character(original_vars))
kept_vars     <- tolower(as.character(var_df[[1]]))

removed_vars <- setdiff(unique(original_vars), unique(kept_vars))

In [9]:
scf_list <- scf_list %>%
  rename_with(tolower) %>%
  select(-any_of(removed_vars))

Exclude the outcome, survey weights, and case IDS from analysis.

In [10]:
exclude <- c("x7557", "yy1", "y1", "x42000", "x42001", "wgt", "five")

pred_vars <- setdiff(names(scf_list), exclude)
pred_vars <- as.character(pred_vars)

vars_na <- pred_vars[vapply(scf_list[, pred_vars, drop = FALSE], anyNA, logical(1))]

for (v in vars_na) {
  scf_list[[paste0(v, "_miss")]] <- as.integer(is.na(scf_list[[v]]))
  }

In [11]:
s1_df <- scf_list[str_sub(scf_list[, 'y1'] ,-1 ,-1) == 1 ,]
s2_df <- scf_list[str_sub(scf_list[, 'y1'] ,-1 ,-1) == 2 ,]
s3_df <- scf_list[str_sub(scf_list[, 'y1'] ,-1 ,-1) == 3 ,]
s4_df <- scf_list[str_sub(scf_list[, 'y1'] ,-1 ,-1) == 4 ,]
s5_df <- scf_list[str_sub(scf_list[, 'y1'] ,-1 ,-1) == 5 ,]

scf_list <- list(s1_df , s2_df , s3_df , s4_df , s5_df)

In [12]:
med_imp <- function(df, vars) {
  num_vars <- vars[vapply(df[, vars, drop = FALSE],
                          \(x) is.numeric(x) || is.integer(x), logical(1))]
  for (v in num_vars) {
    x <- df[[v]]
    if (anyNA(x)) {
      med <- median(x, na.rm = TRUE)
      if (is.finite(med)) df[[v]][is.na(x)] <- med
    }
  }
  df
}

scf_list <- lapply(scf_list, med_imp, vars = pred_vars)

### **LASSO Regression**
We will now run LASSO regression on each imputation using the **glmnet** package, utilizing the survey case weights. Ideally, this would be done using packages such as svyVarSel to fit a LASSO model to a complex survey design object (svydesign). However, the SCF utilizes replicate weights instead of the traditional sampling design with strata and PSUs, meaning we need to use a svrepdesign object instead for analysis. Most methods and literature for complex survey data do not address the differences in design, so we will instead run a weighted LASSO for each imputation. The final set of predictors will be refitted on a svrepdesign object for prediction in a later notebook.

In [14]:
make_xyw <- function(df, xform, yvar="x7557", wvar="x42000") {
  y <- df[[yvar]]
  X <- model.matrix(xform, data = df)[, -1, drop = FALSE]
  w <- df[[wvar]]
  list(X=X, y=y, w=w)
}

Define the predictor variables

In [13]:
pred_vars <- setdiff(names(scf_list[[1]]), exclude)
pred_vars <- as.character(pred_vars)
xform <- reformulate(pred_vars)

xform

~x7398 + x7578 + x7579 + x7019 + x7001 + x7050 + x8020 + x5908 + 
    x7005 + x8023 + x8024 + x7002 + x301 + x7489 + x7556 + x7561 + 
    x7101 + x7102 + x7103 + x7104 + x7105 + x7106 + x7107 + x7108 + 
    x7109 + x7110 + x6849 + x6861 + x6862 + x7562 + x7112 + x7113 + 
    x7114 + x7115 + x7116 + x7117 + x7118 + x7119 + x7120 + x7121 + 
    x6865 + x6866 + x6867 + x401 + x402 + x403 + x405 + x406 + 
    x433 + x434 + x435 + x436 + x437 + x438 + x439 + x440 + x441 + 
    x407 + x408 + x7585 + x409 + x7583 + x305 + x7592 + x7593 + 
    x7122 + x7126 + x7594 + x7648 + x7582 + x7973 + x411 + x412 + 
    x413 + x414 + x7132 + x415 + x416 + x417 + x418 + x7500 + 
    x6648 + x6649 + x7974 + x419 + x420 + x421 + x7976 + x425 + 
    x426 + x427 + x432 + x442 + x443 + x7577 + x7576 + x7575 + 
    x501 + x502 + x503 + x504 + x505 + x506 + x507 + x508 + x509 + 
    x510 + x511 + x513 + x517 + x518 + x519 + x520 + x521 + x522 + 
    x523 + x526 + x601 + x602 + x603 + x604 + x608 + x606 + x607 + 

Weighted LASSO on each imputation. Although the outcome variable (x7557) is ordinal, glmnet does not contain built-in families for ordinal regression. Instead, we will treat x7557 as approximately continuous using "gaussian" in the family argument.

In [15]:
set.seed(42)

cvfits <- lapply(scf_list, function(df) {
  dat <- make_xyw(df, xform, yvar="x7557", wvar="x42000")

  cv.glmnet(
    x = dat$X,
    y = dat$y,
    weights = dat$w,
    family = "gaussian",
    alpha = 1,
    nfolds = 10,
    standardize = TRUE
  )
})

Identify variables selected in at least 4/5 of the imputations.

In [16]:
sel_each <- lapply(cvfits, function(fit) {
  b <- coef(fit, s = "lambda.1se")
  setdiff(rownames(b)[as.vector(b != 0)], "(Intercept)")
})

freq <- table(unlist(sel_each))

sel_vars <- names(freq)[freq >= 4]
sel_vars

[1] "age"       "assetcat"  "bshopmodr" "foodaway"  "hbrok"     "hbus"     
 [7] "hequity"   "hhsex"     "hstocks"   "internet"  "ishopnone" "knowl"    
[13] "lifecl"    "reteq"     "savres8"   "spendmor"  "x1301"     "x3023"    
[19] "x3103"     "x3913"     "x3921"     "x3923"     "x401"      "x414"     
[25] "x435"      "x5711"     "x5822"     "x5824"     "x5825"     "x5913"    
[31] "x6030"     "x6354"     "x6556"     "x6789"     "x6859"     "x7005"    
[37] "x7113"     "x7144"     "x7196"     "x7198"     "x7364"     "x7409"    
[43] "x7422"     "x7489"     "x7509"     "x7556"     "x7558"     "x7561"    
[49] "x7562"     "x7578"     "x7650"     "x7761"     "x7778"     "x8300"

In [17]:
# SCF Variables
write.csv(bind_rows(scf_list), "scf_list.csv", row.names = FALSE)

#Selected features
write.csv(data.frame(sel_vars), "var_names.csv", row.names = FALSE)